In [39]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import PromptTemplate
import os
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda
from pydantic import  BaseModel, Field
from enum import Enum

In [2]:
load_dotenv()

True

In [3]:
openai_model1 = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=2048,
)


In [36]:
google_model2 = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    max_output_tokens=2048,
)


In [37]:
anthropic_model3 = ChatAnthropic(
    model="anthropic/claude-haiku-4.5",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api",
    max_tokens=2048,
)


In [ ]:
class SentimentTypes(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL  = "neutral"

In [11]:
class Feedback(BaseModel):
    sentiment: SentimentTypes = Field(description="The sentiment of the feedback")

In [8]:
parser1 = StrOutputParser()

In [13]:
parser2 = PydanticOutputParser(pydantic_object=Feedback)

In [27]:
prompt1 = PromptTemplate(
    template = 'Classify the sentiment of the following feedback: {feedback}, please respond with the sentiment only. {format_instructions}',
    input_variables = ['feedback'],
    partial_variables = {'format_instructions': parser2.get_format_instructions()}
)

In [28]:
classification_chain = prompt1 | openai_model1 | parser2

In [29]:
result = classification_chain.invoke({'feedback': "I don't know what should i say about this product"}).sentiment

In [31]:
print(result.value)

neutral


In [33]:
positive_prompt = PromptTemplate(
    template = 'Write an professional response to this positive feedback: {feedback}',
    input_variables = ['feedback']
)

In [34]:
negative_prompt = PromptTemplate(
    template="Write an professional response to this negative feedback: {feedback}",
    input_variables=["feedback"],
)


In [56]:
neutral_prompt = PromptTemplate(
    template="Write an professional response to this neutral or other than positive or negative feedback: {feedback}",
    input_variables=["feedback"],
)

In [66]:
branch_chain = RunnableBranch(
    (lambda x: x.sentiment == SentimentTypes.POSITIVE, positive_prompt | openai_model1 | parser1),
    (lambda x: x.sentiment == SentimentTypes.NEGATIVE, negative_prompt | google_model2 | parser1),
    (lambda x: x.sentiment == SentimentTypes.NEUTRAL, neutral_prompt | anthropic_model3 | parser1),
    RunnableLambda(lambda x: "Could not determine sentiment.")
)

In [51]:
final_chain = classification_chain | branch_chain

In [52]:
print(final_chain.invoke({"feedback": "This is a great product!"}))

Subject: Thank You for Your Feedback!

Dear [Recipient's Name],

Thank you so much for your positive feedback! I truly appreciate you taking the time to share your thoughts. It's always encouraging to hear that our efforts are making a positive impact.

We strive to provide the best possible experience, and your kind words serve as a great motivation for our team. If you have any further suggestions or insights, please feel free to share. We are always looking for ways to improve and enhance our services.

Thank you once again for your support!

Best regards,

[Your Name]  
[Your Position]  
[Your Company]  
[Your Contact Information]  


In [53]:
print(final_chain.invoke({"feedback": "This is a terrible product!"}))


Because you didn't include the specific details of the negative feedback, I have created a professional template below. 

You can easily **fill in the bracketed information [like this]** to fit your specific situation. 

***

**Subject:** Regarding your recent experience with [Company/Product Name]

Dear [Customer Name],

Thank you for reaching out and sharing your feedback with us. I am truly sorry to hear that your recent experience did not meet your expectations, especially regarding [mention the specific issue briefly, e.g., our customer service / the delay with your order]. 

At [Company Name], we strive to provide the highest standard of service and products, and it is clear we fell short in this instance. We take feedback like yours very seriously, as it helps us identify areas where we need to improve.

I would love the opportunity to make things right for you. [Choose one: I have processed a full refund for your purchase / I have credited your account / I would like to offer y

In [68]:
print(
    final_chain.invoke(
        {
            "feedback": "The product arrived yesterday and weighs approximately 750 grams and it seems like good value for the price."
        }
    )
)


Subject: Thank You for Your Positive Feedback!

Dear [Recipient's Name],

Thank you very much for taking the time to share your positive feedback. We truly appreciate your kind words and are thrilled to hear that you had a great experience with us.

At [Your Company/Organization Name], we strive to deliver the highest quality service and foster positive relationships with our clients. Your comments serve as a wonderful reminder that our efforts are making a difference.

If there's anything more we can do for you or if you have any further suggestions, please feel free to reach out. We look forward to continuing to serve you!

Best regards,

[Your Name]  
[Your Position]  
[Your Company/Organization Name]  
[Your Contact Information]  
